In [4]:
import pandas as pd
import numpy as np
import rasterio
import os
import glob
from collections import defaultdict


In [ ]:

# ============================
# 1. Load CO2 CSV
# ============================
co2 = pd.read_csv(r"E:\\DownloadData\\co2_nasa\\data_processed\\oco_rice_buffer\\oco_rice_buffer_2024.csv")
co2['date'] = pd.to_datetime(co2['date']).dt.strftime('%Y-%m-%d')

# ============================
# 2. Folder paths
# ============================
folders = {
    "CHIRPS": r"E:\DownloadData\gee\modis_climate_tif\chirps\chirps_precipitation_2024",
    "ERA5": r"E:\DownloadData\gee\modis_climate_tif\era5\era5_2024",
    "MOD16": r"E:\DownloadData\gee\modis_climate_tif\et_pet\et_pet_2024",
    "MCD15A3H": r"E:\DownloadData\gee\modis_climate_tif\fpar_lai\modis_fpar_lai_2024",
    "MOD09GA": r"E:\DownloadData\gee\modis_climate_tif\ndvi_evi\ndvi_evi_2024",
    "MCD19A2": r"E:\DownloadData\gee\modis_climate_tif\optical_depth\aod_2024",
    "MCD18C2": r"E:\DownloadData\gee\modis_climate_tif\par\par_2024",
    "SMAP": r"E:\DownloadData\gee\modis_climate_tif\smap\smap_2024"
}

# ============================
# 3. Định nghĩa bands
# ============================
BAND_CONFIG = {
    "CHIRPS": ["chirps_precipitation"],
    "ERA5": ["temperature_2m", 
             "skin_temperature", 
             "soil_temperature_L1",
             "soil_water_L1", 
             "surface_solar_radiation", 
             "total_precipitation_era5",
             "lai_low_veg", "wind_u_10m", 
             "wind_v_10m", 
             "dewpoint_temp_2m", 
             "surface_pressure"],
    "MOD16": ["ET", "LE", "PET", "PLE"],
    "MCD15A3H": ["FPAR", "LAI"],
    "MOD09GA": ["NDVI", "EVI", "LSWI", "NDWI_McFeeters"],
    "MCD19A2": ["AOD_047um", "AOD_055um"],
    "MCD18C2": ["GMT_0000_PAR", "GMT_0300_PAR", "GMT_0600_PAR", "GMT_0900_PAR"],
    "SMAP": ["sm_surface"]
}

# ============================
# 4. Hàm trích xuất BATCH (mở file 1 lần)
# ============================
def extract_batch(tif_file, coords, band_names):
    """
    coords: list of (idx, lat, lon)
    band_names: list of variable names
    Returns: dict {idx: {var: value}}
    """
    results = defaultdict(dict)
    
    with rasterio.open(tif_file) as src:
        nodata = src.nodata
        
        # Đọc tất cả bands cần thiết
        bands_data = [src.read(i+1) for i in range(len(band_names))]
        
        for idx, lat, lon in coords:
            try:
                row, col = src.index(lon, lat)
                for band_idx, var_name in enumerate(band_names):
                    value = bands_data[band_idx][row, col]
                    results[idx][var_name] = float(value) if value != nodata else np.nan
            except IndexError:
                for var_name in band_names:
                    results[idx][var_name] = np.nan
    
    return results

# ============================
# 5. Map file theo ngày
# ============================
file_maps = {}
for key, path in folders.items():
    files = glob.glob(os.path.join(path, "*.tif"))
    file_maps[key] = {os.path.basename(f)[-14:-4]: f for f in files}

# ============================
# 6. Tạo cột rỗng
# ============================
all_vars = []
for bands in BAND_CONFIG.values():
    all_vars.extend(bands)

for v in all_vars:
    co2[v] = np.nan

# ============================
# 7. Nhóm điểm theo ngày
# ============================
date_groups = co2.groupby('date').groups  # {date: [idx1, idx2, ...]}

# ============================
# 8. Xử lý theo từng dataset
# ============================
for dataset_name, band_names in BAND_CONFIG.items():
    print(f"⏳ Đang xử lý {dataset_name}...")
    
    for date, indices in date_groups.items():
        if date not in file_maps[dataset_name]:
            continue
        
        tif_file = file_maps[dataset_name][date]
        
        # Lấy tọa độ của các điểm trong ngày này
        coords = [(idx, co2.at[idx, 'latitude'], co2.at[idx, 'longitude']) 
                  for idx in indices]
        
        # Trích xuất BATCH
        results = extract_batch(tif_file, coords, band_names)
        
        # Gán kết quả vào DataFrame
        for idx, values in results.items():
            for var_name, value in values.items():
                co2.at[idx, var_name] = value
    
    print(f"✓ Hoàn thành {dataset_name}")

# ============================
# 9. Xuất CSV
# ============================
output_file = r"E:\DownloadData\gee\output\co2_modis_climate_2024.csv"
co2.to_csv(output_file, index=False)
print("✔ Hoàn tất, file lưu tại:", output_file)

⏳ Đang xử lý CHIRPS...
✓ Hoàn thành CHIRPS
⏳ Đang xử lý ERA5...
✓ Hoàn thành ERA5
⏳ Đang xử lý MOD16...
✓ Hoàn thành MOD16
⏳ Đang xử lý MCD15A3H...
✓ Hoàn thành MCD15A3H
⏳ Đang xử lý MOD09GA...
✓ Hoàn thành MOD09GA
⏳ Đang xử lý MCD19A2...
✓ Hoàn thành MCD19A2
⏳ Đang xử lý MCD18C2...
✓ Hoàn thành MCD18C2
⏳ Đang xử lý SMAP...
✓ Hoàn thành SMAP
✔ Hoàn tất, file lưu tại: E:\DownloadData\gee\output\co2_modis_climate_2024.csv
